In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["REDIS_URL"] = os.getenv("REDIS_URL")

print("✓ REDIS_URL cargada")

✓ REDIS_URL cargada


In [4]:
import pandas as pd
csv_path = "../app/data/PRSA_data_2010.1.1-2014.12.31.csv"
df = pd.read_csv(csv_path)
df['date'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
print("✓ CSV cargado y procesado")
print(df.head())

✓ CSV cargado y procesado
   No  year  month  day  hour  pm2.5  DEWP  TEMP    PRES cbwd    Iws  Is  Ir  \
0   1  2010      1    1     0    NaN   -21 -11.0  1021.0   NW   1.79   0   0   
1   2  2010      1    1     1    NaN   -21 -12.0  1020.0   NW   4.92   0   0   
2   3  2010      1    1     2    NaN   -21 -11.0  1019.0   NW   6.71   0   0   
3   4  2010      1    1     3    NaN   -21 -14.0  1019.0   NW   9.84   0   0   
4   5  2010      1    1     4    NaN   -20 -12.0  1018.0   NW  12.97   0   0   

                 date  
0 2010-01-01 00:00:00  
1 2010-01-01 01:00:00  
2 2010-01-01 02:00:00  
3 2010-01-01 03:00:00  
4 2010-01-01 04:00:00  


In [7]:
import pandas as pd

df = pd.read_csv("../app/data/PRSA_data_2010.1.1-2014.12.31.csv")
df_2014 = df[df["year"] == 2014].copy()
df_2014["year"] = 2026
df_2014.to_csv("../app/data/dummies_2026.csv", index=False)
print(df_2014.head())

          No  year  month  day  hour  pm2.5  DEWP  TEMP    PRES cbwd     Iws  \
35064  35065  2026      1    1     0   24.0   -20   7.0  1014.0   NW  143.48   
35065  35066  2026      1    1     1   53.0   -20   7.0  1013.0   NW  147.50   
35066  35067  2026      1    1     2   65.0   -20   6.0  1013.0   NW  151.52   
35067  35068  2026      1    1     3   70.0   -20   6.0  1013.0   NW  153.31   
35068  35069  2026      1    1     4   79.0   -18   3.0  1012.0   cv    0.89   

       Is  Ir  
35064   0   0  
35065   0   0  
35066   0   0  
35067   0   0  
35068   0   0  


In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta



NameError: name '__file__' is not defined

In [3]:
import joblib
from pathlib import Path

BASE_PATH = Path("..").resolve()
modelo = joblib.load(BASE_PATH / "app/models/model_xgb_h1.joblib")

# Ver qué espera el modelo
print("Número de features esperadas:")
print(modelo.n_features_in_)

print("\nNombres de features (si están guardados):")
print(modelo.feature_names_in_)

Número de features esperadas:
58

Nombres de features (si están guardados):
['pm2.5' 'DEWP' 'TEMP' 'PRES' 'Iws' 'Is' 'Ir' 'cbwd_NE' 'cbwd_NW'
 'cbwd_SE' 'cbwd_cv' 'month_1' 'month_2' 'month_3' 'month_4' 'month_5'
 'month_6' 'month_7' 'month_8' 'month_9' 'month_10' 'month_11' 'month_12'
 'pm2.5_(t-1)' 'pm2.5_(t-2)' 'pm2.5_(t-3)' 'pm2.5_(t-4)' 'pm2.5_(t-5)'
 'pm2.5_(t-6)' 'pm2.5_(t-7)' 'DEWP_(t-1)' 'DEWP_(t-2)' 'DEWP_(t-3)'
 'DEWP_(t-4)' 'DEWP_(t-5)' 'DEWP_(t-6)' 'DEWP_(t-7)' 'TEMP_(t-1)'
 'TEMP_(t-2)' 'TEMP_(t-3)' 'TEMP_(t-4)' 'TEMP_(t-5)' 'TEMP_(t-6)'
 'TEMP_(t-7)' 'PRES_(t-1)' 'PRES_(t-2)' 'PRES_(t-3)' 'PRES_(t-4)'
 'PRES_(t-5)' 'PRES_(t-6)' 'PRES_(t-7)' 'Iws_(t-1)' 'Iws_(t-2)'
 'Iws_(t-3)' 'Iws_(t-4)' 'Iws_(t-5)' 'Iws_(t-6)' 'Iws_(t-7)']


In [1]:
print("end")

end


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from dotenv import load_dotenv
load_dotenv()

from app.services.redismanager import redismanager
from datetime import datetime, timedelta
import pandas as pd


def obtener_contexto_pronostico():
    """
    Lee el último pronóstico de Redis y lo convierte en contexto para el LLM.
    
    Returns:
        str: Contexto formateado o string vacío si no hay datos.
    """
    try:
        ultima_pred = redismanager.obtener_prediction()
        
        if ultima_pred is not None and not ultima_pred.empty:
            fecha = ultima_pred.iloc[0]['date']
            contexto = f"\n\nÚltimo pronóstico de PM2.5 (fecha de inferencia: {fecha}):\n"
            for h in range(1, 8):
                valor = ultima_pred.iloc[0][f'pm2.5_(t+{h})']
                contexto += f"- h{h}: {valor} µg/m³\n"
            return contexto
    except Exception as e:
        print(f"Error al obtener contexto: {e}")
    
    return ""

print(obtener_contexto_pronostico())